# Descargar tabla de atributos: **Humedales Urbanos Declarados** (Visor SIMBIO)

Este notebook consulta el servicio ArcGIS REST del MMA asociado a la capa **Humedales Urbanos Declarados** y descarga su **tabla de atributos** en CSV/XLSX.

Fuente (ArcGIS REST): `SIMBIO/SIMBIO_HUMEDALES` → Layer `1`.


In [1]:
# !pip -q install pandas requests openpyxl

import math
import time
import requests
import pandas as pd


## 1) Configurar endpoint y parámetros

In [2]:
# Endpoint ArcGIS REST (Layer 1: Humedales Urbanos Declarados)
LAYER_URL = "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_HUMEDALES/MapServer/1/query"

# Parámetros base para consulta
BASE_PARAMS = {
    "f": "json",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
}

# Para evitar bloqueos por cache/CDN a veces ayuda un user-agent explícito
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; BIODATA/1.0; +https://ieb-chile.cl)"
}


## 2) Consultar cantidad de registros (para paginar)

In [3]:
def get_count():
    params = dict(BASE_PARAMS)
    params.update({
        "returnCountOnly": "true"
    })
    r = requests.get(LAYER_URL, params=params, headers=HEADERS, timeout=60)
    r.raise_for_status()
    data = r.json()
    if "count" not in data:
        raise RuntimeError(f"Respuesta inesperada al pedir count: {data}")
    return int(data["count"])

total = get_count()
total


137

## 3) Descargar todos los atributos (paginación por `resultOffset`)

In [4]:
def fetch_page(offset: int, page_size: int = 2000) -> pd.DataFrame:
    params = dict(BASE_PARAMS)
    params.update({
        "resultOffset": offset,
        "resultRecordCount": page_size,
        "orderByFields": "OBJECTID ASC",  # orden estable
    })
    r = requests.get(LAYER_URL, params=params, headers=HEADERS, timeout=120)
    r.raise_for_status()
    data = r.json()

    if "error" in data:
        raise RuntimeError(f"Error ArcGIS: {data['error']}")
    feats = data.get("features", [])
    rows = [f.get("attributes", {}) for f in feats]
    return pd.DataFrame(rows)

# Descarga paginada
PAGE_SIZE = 2000  # el servicio reporta MaxRecordCount=2000
pages = math.ceil(total / PAGE_SIZE)

dfs = []
for i in range(pages):
    offset = i * PAGE_SIZE
    df_i = fetch_page(offset, PAGE_SIZE)
    print(f"Página {i+1}/{pages}: {len(df_i)} filas (offset={offset})")
    dfs.append(df_i)

    # micro-pausa para ser amable con el servicio
    time.sleep(0.2)

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
df.shape


Página 1/1: 137 filas (offset=0)


(137, 11)

## 4) Chequeos rápidos

In [5]:
df.head()


,COD_HUM_M,NOMBRE,COMUNA,PROVINCIA,REGION,HECTAREAS,PROCESO,RESOLUCION,URL_RES_BCN,OBJECTID,URL_SIMBIO
0,HU-0012,Aguada La Chimba,ANTOFAGASTA,ANTOFAGASTA,REGIÓN DE ANTOFAGASTA,1.005898,O,R.E. N°0787/2021,http://bcn.cl/2ul3k,1,https://sistemahumedales.mma.gob.cl/OficioHU/D...
1,HU-0031,Baños Morales,SAN JOSÉ DE MAIPO,CORDILLERA,REGIÓN METROPOLITANA DE SANTIAGO,2.809060,O,R.E. N°0990/2021,http://bcn.cl/2svqw,2,https://sistemahumedales.mma.gob.cl/OficioHU/D...
2,HU-0029,Bucalemu,PAREDONES,CARDENAL CARO,REGIÓN DEL LIBERTADOR GENERAL BERNARDO O'HIGGINS,279.376802,O,R.E. N°0982/2021,http://bcn.cl/2ul4c,3,https://sistemahumedales.mma.gob.cl/OficioHU/D...
3,HU-0010,Circuito Humedales Pudeto Bajo,ANCUD,CHILOÉ,REGIÓN DE LOS LAGOS,37.247461,M,RE N° 784/ 2021,http://bcn.cl/2rcl5,4,https://sistemahumedales.mma.gob.cl/HumedalesU...
4,HU-0027,El Avellano,LOS ÁNGELES,BIOBÍO,REGIÓN DEL BIOBÍO,6.617904,O,RE N° 921/ 2021,http://bcn.cl/2sspa,5,https://sistemahumedales.mma.gob.cl/OficioHU/D...


In [16]:
df["anio_publicacion"] = (
    df["RESOLUCION"]
    .str.extract(r"/\s*(\d{4})", expand=False)
    .astype("Int64")
)

df["anio_publicacion"].isna().any()
df["anio_publicacion"].agg(["min", "max"])


min    2021
max    2025
Name: anio_publicacion, dtype: int64

In [19]:
df["HECTAREAS"] = pd.to_numeric(df["HECTAREAS"], errors="coerce")

tabla_anual = (
    df
    .groupby("anio_publicacion", as_index=False)["HECTAREAS"]
    .sum()
    .sort_values("anio_publicacion")
)

tabla_anual["hectareas_acumuladas"] = (
    tabla_anual["HECTAREAS"].cumsum()
)
tabla_anual

,anio_publicacion,HECTAREAS,hectareas_acumuladas
0,2021,5471.973004,5471.973004
1,2022,2026.481850,7498.454854
2,2023,2020.685909,9519.140762
3,2024,1197.919144,10717.059907
4,2025,6131.217129,16848.277036


In [20]:
#escribir tabla anual a Excel
tabla_anual.to_excel("I_3_humedales_urbanos_tabla_anual.xlsx", index=False)